**10_streamlit_dashboard.ipynb: Streamlit Real-Time Dashboard & Interactive Fraud Risk Interface**

**Multimodal Financial Fraud / Risk Detector**  
This notebook documents and verifies **Phase 11: Streamlit Real-Time Dashboard**.  
The dashboard provides a decoupled web-based graphical interface for fraud analysts, risk officers, and auditors, consuming the underlying `FraudInferencePipeline` implemented in Phase 10.

```
STREAMLIT USER INTERACTION ARCHITECTURE
                    Streamlit UI (app/app.py)
                               │
               ┌───────────────┴───────────────┐
               ▼                               ▼
     Single Transaction Scoring       Batch Audit & Clearing
               │                               │
               └───────────────┬───────────────┘
                               ▼
            FraudInferencePipeline (src.inference)
                               │
            ┌──────────────────┼──────────────────┐
            ▼                  ▼                  ▼
    TabularPreprocessor  Hybrid Neural Net   Counterfactual
    (models/preproc)     (models/hybrid.pt)   Explainability
```

**Core Capabilities**:
1. **Single Transaction Analysis**: Interactive guided form and JSON editor with preset archetypes (Legitimate, High-Risk Anomaly, Cross-Border Velocity Surge).
2. **Interactive Risk Policy**: Dynamic threshold slider ($	heta \in [0.05, 0.95]$) simulating risk tier transitions in real time.
3. **Local Explainability Evidence**: Sensitivity charts identifying features elevating fraud risk relative to empirical baseline imputation.
4. **Batch Fraud Clearing & Audit**: File uploader and built-in 500-transaction benchmark batch evaluation with high-throughput inference (>1,500 txn/sec) and CSV report download.


**1. Setup & Environment**

We verify imports and validate that the Streamlit application file `app/app.py` is present and error-free.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import streamlit as st
from streamlit.testing.v1 import AppTest

app_file = PROJECT_ROOT / "app" / "app.py"
print(f"Streamlit Version: {st.__version__}")
print(f"App Script File:   {app_file.name} (Exists: {app_file.exists()})")


Streamlit Version: 1.64.0
App Script File:   app.py (Exists: True)


**2. Headless Verification via Streamlit AppTest Framework**

We run `streamlit.testing.v1.AppTest` against `app/app.py` to programmatically verify that all widgets, cached resources, and tabs mount without unhandled exceptions.


In [2]:
at = AppTest.from_file(str(app_file), default_timeout=30)
at.run()

print("=== AppTest Initial Mount Verification ===")
print(f"Exceptions Raised: {len(at.exception)}")
print(f"Sidebar Title:     {at.sidebar.title[0].value if at.sidebar.title else 'None'}")
print(f"Tabs Mounted:      {[t.label for t in at.tabs] if hasattr(at, 'tabs') else 'Present'}")
assert len(at.exception) == 0, f"AppTest raised unexpected exceptions: {at.exception}"


2026-10-02 12:58:25.535 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


=== AppTest Initial Mount Verification ===
Exceptions Raised: 0
Sidebar Title:     Fraud Risk Engine
Tabs Mounted:      ['🔍 Real-Time Transaction Scoring', '📊 Batch Fraud Clearing & Audit', '📖 Model Architecture & Evaluation']


**3. Automated Simulation: Scoring Live Single Transaction**

We simulate a user clicking the **Score Live Transaction** button, verifying that `FraudInferencePipeline.predict()` executes, returning the calibrated probability, decision badge, latency, and Altair sensitivity chart.


In [3]:
# Click Score Live Transaction (button[0])
score_btn = at.button[0]
print(f"Triggering Action: '{score_btn.label}'")
score_btn.click().run()

print(f"Execution Exceptions: {len(at.exception)}")
assert len(at.exception) == 0, f"Execution failed on score button: {at.exception}"
print("Single transaction scoring and explainability rendered with 0 exceptions.")


Triggering Action: '🚀 Score Live Transaction'


2026-10-02 12:58:34.625 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.


2026-10-02 12:58:34.668 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


Execution Exceptions: 0
Single transaction scoring and explainability rendered with 0 exceptions.


**4. Automated Simulation: Batch Transaction Clearing Audit**

We test Tab 2 by triggering the **Load 500-Transaction Sample Batch** button and running **Execute Batch Risk Scoring**. This exercises `predict_batch()` across 500 records.


In [4]:
# Click Load Sample Batch (button[1])
sample_batch_btn = at.button[1]
print(f"Triggering Action: '{sample_batch_btn.label}'")
sample_batch_btn.click().run()
assert len(at.exception) == 0

# Click Execute Batch Scoring (button[2])
exec_batch_btn = at.button[2]
print(f"Triggering Action: '{exec_batch_btn.label}'")
exec_batch_btn.click().run()

print(f"Batch Execution Exceptions: {len(at.exception)}")
assert len(at.exception) == 0, f"Batch execution failed: {at.exception}"
print("Batch scoring audit completed with 0 exceptions.")


2026-10-02 12:58:34.788 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.


2026-10-02 12:58:34.792 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


2026-10-02 12:58:34.827 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


Triggering Action: '📂 Load 500-Transaction Sample Batch'


Triggering Action: '⚡ Execute Batch Risk Scoring'


2026-10-02 12:58:34.983 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.


2026-10-02 12:58:34.987 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


Batch Execution Exceptions: 0
Batch scoring audit completed with 0 exceptions.


**5. Launch Instructions for Local & Cloud Deployment**

To launch the interactive dashboard in a web browser:

```bash
# Activate virtual environment
.venv\Scripts\activate   # Windows
# source .venv/bin/activate # Linux/macOS

# Run Streamlit Application
streamlit run app/app.py
```

The application will be accessible at `http://localhost:8501`.

**Key Architectural Takeaways**:
- **Strict Decoupling**: `app/app.py` contains zero training or preprocessing fitting logic. It purely consumes `src.inference.FraudInferencePipeline`.
- **Zero Leakage**: Scalers and embedding vocabularies are loaded from frozen artifacts.
- **Explainability**: Attribution values reflect local model sensitivity, protecting against causal over-claims.
